# MNIST PCA Benchmark

Downloads MNIST, reduces to PCA components, runs CrossCat inference,
and evaluates structure discovery and held-out imputation quality.

**Benchmark details:**
- 2000 MNIST samples, stratified (200 per digit)
- 20 PCA components (continuous features)
- 2 chains x 30 sweeps with 5% held-out cells
- Evaluates: view discovery, digit-cluster contingency, imputation MAE

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    # Checkout branch: try local first, then create from remote tracking branch
    checkout = subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR)
    if checkout.returncode != 0:
        subprocess.run(
            ["git", "checkout", "-b", BRANCH, f"origin/{BRANCH}"], cwd=WORKDIR, check=True
        )
    else:
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"], check=True)

import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from benchmarks.utils import create_results_dir, plot_convergence, plot_z_matrix, save_metrics
from crosscat import (
    ColumnType,
    collect_diagnostics,
    initialize,
    pack_state,
    packed_dependence_matrix,
    packed_evaluate_imputation,
    packed_gibbs_sweep,
    random_holdout_mask,
    unpack_state,
)

print(f"JAX {jax.__version__} | Backend: {jax.default_backend()} | Devices: {jax.devices()}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Configuration

In [ ]:
N_SAMPLES = 2000
N_COMPONENTS = 20
N_SWEEPS = 30
N_CHAINS = 2
HOLDOUT_FRACTION = 0.05
DIAG_INTERVAL = 10
SEED = 42

## 3. Fetch and Reduce MNIST

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("Fetching MNIST dataset...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="liac-arff")
images_all = mnist.data.astype(np.float32)
labels_all = mnist.target.astype(np.int32)

# Stratified subsample
rng = np.random.default_rng(SEED)
per_digit = N_SAMPLES // 10
indices = []
for digit in range(10):
    digit_idx = np.where(labels_all == digit)[0]
    chosen = rng.choice(digit_idx, size=min(per_digit, len(digit_idx)), replace=False)
    indices.append(chosen)
indices = np.concatenate(indices)
rng.shuffle(indices)
images, labels = images_all[indices], labels_all[indices]

# PCA reduce
print(f"Reducing 784 dims to {N_COMPONENTS} PCA components...")
scaled = StandardScaler().fit_transform(images)
data_np = PCA(n_components=N_COMPONENTS).fit_transform(scaled).astype(np.float32)
data_jax = jnp.array(data_np)
col_types = [ColumnType.CONTINUOUS] * N_COMPONENTS

print(f"Data: {data_jax.shape}")
unique, counts = np.unique(labels, return_counts=True)
print(f"Digits: {dict(zip(unique, counts, strict=True))}")

## 4. Create Holdout Mask

In [ ]:
rng_key = jax.random.key(SEED)
k_mask, k_init = jax.random.split(rng_key)
mask = random_holdout_mask(k_mask, N_SAMPLES, N_COMPONENTS, HOLDOUT_FRACTION)
data_masked = jnp.where(mask, jnp.nan, data_jax)
print(f"Held out {int(mask.sum())} cells ({HOLDOUT_FRACTION * 100:.0f}%)")

## 5. Run Gibbs Inference

In [ ]:
init_keys = jax.random.split(k_init, N_CHAINS)
states = []
all_chain_metrics = []

for chain_idx in range(N_CHAINS):
    print(f"\n--- Chain {chain_idx + 1}/{N_CHAINS} ---")
    k_i, k_sweep = jax.random.split(init_keys[chain_idx])
    state = initialize(k_i, data_masked, col_types).state
    packed = pack_state(state)
    chain_metrics = []
    t0 = time.time()

    sweep = 0
    while sweep < N_SWEEPS:
        batch = min(DIAG_INTERVAL, N_SWEEPS - sweep)
        k_sweep, subkey = jax.random.split(k_sweep)
        packed = packed_gibbs_sweep(subkey, packed, data_masked, n_sweeps=batch)
        sweep += batch
        state_tmp = unpack_state(packed, col_types, data=data_masked)
        diag = collect_diagnostics(state_tmp, data_masked)
        chain_metrics.append({"sweep": sweep, **diag})
        del state_tmp
        if sweep % 10 == 0 or sweep == N_SWEEPS:
            print(f"  Sweep {sweep:3d}/{N_SWEEPS}: log_joint={diag['log_joint']:.0f}")

    state = unpack_state(packed, col_types, data=data_masked)
    elapsed = time.time() - t0
    print(f"  Time: {elapsed:.1f}s ({elapsed / N_SWEEPS:.2f}s/sweep)")
    states.append(state)
    all_chain_metrics.append(chain_metrics)

## 6. Evaluation

In [ ]:
packed_states = [pack_state(s) for s in states]
final_packed = packed_states[-1]
final_state = states[-1]

print(f"Views discovered: {final_state.n_views}")
n_clusters = [int(jnp.max(v.row_assignments)) + 1 for v in final_state.views]
print(f"Clusters per view: {n_clusters}")

# Held-out imputation
print("\nEvaluating held-out imputation...")
k_eval = jax.random.key(SEED + 1)
imputation = packed_evaluate_imputation(final_packed, data_jax, mask, col_types, rng_key=k_eval)
print(f"Imputation MAE: {imputation['mae']:.4f}")
print(f"Mean log-lik: {imputation['mean_log_lik']:.4f}")

results = {
    "n_views": final_state.n_views,
    "n_clusters_per_view": n_clusters,
    "imputation_mae": imputation["mae"],
    "imputation_mean_log_lik": imputation["mean_log_lik"],
}

## 7. Convergence Plot

In [ ]:
def average_chain_metrics(all_chain_metrics):
    n_points = len(all_chain_metrics[0])
    avg = []
    for i in range(n_points):
        combined = {"sweep": all_chain_metrics[0][i]["sweep"]}
        keys = [k for k in all_chain_metrics[0][i] if k != "sweep"]
        for key in keys:
            vals = [
                c[i].get(key) for c in all_chain_metrics if isinstance(c[i].get(key), (int, float))
            ]
            if vals:
                combined[key] = sum(vals) / len(vals)
        avg.append(combined)
    return avg


results_dir = create_results_dir("mnist")
avg_metrics = average_chain_metrics(all_chain_metrics)
plot_convergence(avg_metrics, results_dir, show_log_joint=True)

from IPython.display import Image, display

display(Image(filename=str(results_dir / "convergence.png")))

## 8. Dependence Matrix (Z-matrix)

In [ ]:
z_matrix = packed_dependence_matrix(packed_states)
plot_z_matrix(z_matrix, results_dir, col_labels=[f"PC{i + 1}" for i in range(N_COMPONENTS)])
display(Image(filename=str(results_dir / "z_matrix.png")))

## 9. Digit-Cluster Contingency

In [ ]:
# Build contingency table
view_sizes = [len(v.column_indices) for v in final_state.views]
main_view = final_state.views[int(np.argmax(view_sizes))]
row_assign = np.array(main_view.row_assignments)
n_clust = int(row_assign.max()) + 1

contingency = np.zeros((10, n_clust), dtype=np.int32)
for digit in range(10):
    for c in range(n_clust):
        contingency[digit, c] = int(np.sum(row_assign[labels == digit] == c))

# Plot
row_sums = contingency.sum(axis=1, keepdims=True)
normalized = contingency / np.maximum(row_sums, 1)

fig, ax = plt.subplots(figsize=(max(8, n_clust * 0.8 + 2), 6))
im = ax.imshow(normalized, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Fraction of digit in cluster")
ax.set_xticks(range(n_clust))
ax.set_xticklabels([f"C{i}" for i in range(n_clust)])
ax.set_yticks(range(10))
ax.set_yticklabels([str(d) for d in range(10)])
ax.set_xlabel("Cluster")
ax.set_ylabel("Digit")
ax.set_title("Digit-Cluster Correspondence (Main View)")
for i in range(10):
    for j in range(n_clust):
        if contingency[i, j] > 0:
            ax.text(
                j,
                i,
                str(contingency[i, j]),
                ha="center",
                va="center",
                fontsize=7,
                color="white" if normalized[i, j] > 0.5 else "black",
            )
plt.tight_layout()
fig.savefig(results_dir / "digit_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Save Results

In [ ]:
import shutil

save_metrics(
    {
        **results,
        "config": {
            "n_samples": N_SAMPLES,
            "n_components": N_COMPONENTS,
            "n_sweeps": N_SWEEPS,
            "n_chains": N_CHAINS,
            "holdout_fraction": HOLDOUT_FRACTION,
            "seed": SEED,
        },
        "per_sweep": avg_metrics,
    },
    results_dir,
)

print(f"Results saved to {results_dir}/")
archive = Path("benchmarks/results/mnist_pca_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"Archived to {archive}.tar.gz")